In [1]:
from pathlib import Path

import tiktoken
from pandas import DataFrame
from tiktoken import Encoding

from reverse_se.graph.arcan import ArcanCSV

In [2]:
def simple_architecture(name, graph_path):
    nodes, edges, packagage_files = ArcanCSV(
        name=name, path=graph_path, package_only=True
    ).graph
    print("Num of nodes", len(nodes))
    str_nodes = nodes.to_csv(
        index=False,
        float_format="%.2e",
        sep=" ",
    )
    str_edges = edges.to_csv(index=False, sep=" ", columns=["fromId", "toId", "Weight"])
    if packagage_files is not None:
        df: DataFrame = packagage_files.copy()
        df["files"] = df["files"].apply(
            lambda x: ";".join(x) if isinstance(x, list) else ""
        )
        str_package_files = df.to_csv(index=False, sep=" ")

    encoding: Encoding = tiktoken.encoding_for_model("gpt-5-mini")
    tokens: list[int] = encoding.encode(
        f"{str_nodes}\n{str_edges}\n{str_package_files}"
    )
    nodes_tokens: list[int] = encoding.encode(str_nodes)
    edges_tokens: list[int] = encoding.encode(str_edges)
    package_files_tokens: list[int] = (
        encoding.encode(str_package_files) if packagage_files is not None else []
    )
    res = {"nodes": str_nodes, "edges": str_edges, "package-files": str_package_files}
    count: dict[str, int] = {
        "n_tokens": len(tokens),
        "n_nodes_tokens": len(nodes_tokens),
        "n_edge_tokens": len(edges_tokens),
        "n_package-files_tokens": len(package_files_tokens),
    }

    return res, count

In [3]:
BASE_PATH = Path("../data")
project = "hv-agent-service"
graph_path: str = str(BASE_PATH / "raw" / "arcanOutput" / project)

In [4]:
from pprint import pprint

res, count = simple_architecture(project, graph_path)
pprint(res)

Num of nodes 20
{'edges': 'fromId toId Weight\n'
          '2202 2106 2\n'
          '3326 2380 33\n'
          '2843 2380 20\n'
          '3687 2380 21\n'
          '3687 687 7\n'
          '590 2106 4\n'
          '2843 2765 2\n'
          '687 2380 20\n'
          '3687 1354 1\n'
          '3229 2106 4\n'
          '3326 1354 4\n'
          '2765 2380 8\n'
          '3326 2843 2\n'
          '2202 590 2\n'
          '3687 3326 7\n'
          '2202 2852 2\n'
          '1354 2380 4\n'
          '2106 2202 1\n'
          '3326 2765 2\n'
          '687 3326 4\n'
          '2852 2106 4\n'
          '2202 3229 2\n',
 'nodes': 'ComponentType FanIn FanOut LinesOfCode PageRank name vertexId\n'
          'PACKAGE 0 0 275 1.79e-02 '
          'packages.hv-framework.tests.integration.a2a 0\n'
          'PACKAGE 0 0 315 1.79e-02 packages.hv-framework.tests.unit.llm 1032\n'
          'PACKAGE 4 1 52 2.15e-01 src.hv_agent_service 2106\n'
          'PACKAGE 1 4 39 2.00e-01 src.hv_agent_service.hive